<a href="https://colab.research.google.com/github/amadisamantha-arch/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/amadisamantha-arch/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df)} rows")

Loaded 30000 rows


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Random Forest classifier.

Why it fits: my lane (Refresh/Content Opportunity Scoring) needs a ranked
output, not just a binary label — Precision@K is what matters, since a
reviewer only works through a limited list. Random Forest handles the mix
of numeric and categorical signals in this data without heavy preprocessing,
captures non-linear interactions a single hand-written rule can't (e.g. the
staleness x visibility x CTR-gap interactions my baseline approximated with
fixed thresholds), and gives permutation importance for free, which the
"errors and interpretation" section needs. Logistic Regression was
considered but is less likely to capture the threshold-like interactions my
baseline already hinted at (e.g. staleness only matters combined with real
visibility, not on its own).

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split: client-holdout (grouped by client_id), 75/25.

Why this is honest for this question: pages from the same client tend to
share patterns (site-wide staleness, template CTR behavior, seasonal
demand). A random row-level split could let the model see one client's
pages in training and other pages from that same client in testing,
inflating the score by memorizing client-specific quirks rather than
learning generalizable signal. A client-holdout split tests whether the
model generalizes to clients it has never seen at all, which matches how
this would actually be used — scoring pages for a client whose data the
model may not have trained on.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"Train: {len(train_df)} rows, {train_df['client_id'].nunique()} clients")
print(f"Test: {len(test_df)} rows, {test_df['client_id'].nunique()} clients")
print(f"Client overlap between train/test: {len(set(train_df['client_id']) & set(test_df['client_id']))}")

Train: 22885 rows, 24 clients
Test: 7115 rows, 8 clients
Client overlap between train/test: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Rebuild baseline score on the TEST set only, for a fair comparison
tier_expected_ctr = df[df['impressions_90d'] >= 100].groupby('position_tier')['ctr'].mean().to_dict()

def baseline_score(frame):
    stale_and_visible = ((frame['days_since_last_update'] >= 90) & (frame['impressions_90d'] >= 250)).astype(int)
    expected_ctr = frame['position_tier'].map(tier_expected_ctr)
    ctr_gap = (expected_ctr - frame['ctr']).clip(lower=0)
    ctr_below_tier = (frame['position_tier'].isin(['page_1', 'top_3', 'striking']) & (ctr_gap > 0.05)).astype(int)
    return 0.6 * stale_and_visible * np.log1p(frame['impressions_90d']) + 0.4 * ctr_below_tier * ctr_gap * 100

test_df['baseline_score'] = baseline_score(test_df)
y_test = (test_df['trend_direction'] == 'down').astype(int)

# Train the model
features = ['impressions_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update',
            'avg_position', 'ctr', 'word_count']
features = [f for f in features if f in df.columns]

X_train = train_df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_train = (train_df['trend_direction'] == 'down').astype(int)
X_test = test_df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1)
model.fit(X_train, y_train)
model_scores = model.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    'method': ['baseline (Week 4 rule)', 'random_forest'],
    'Precision@20': [
        precision_at_k(test_df['baseline_score'], y_test, 20),
        precision_at_k(model_scores, y_test, 20)
    ],
    'Precision@50': [
        precision_at_k(test_df['baseline_score'], y_test, 50),
        precision_at_k(model_scores, y_test, 50)
    ],
})
print(results)

                   method  Precision@20  Precision@50
0  baseline (Week 4 rule)          0.75          0.74
1           random_forest          0.60          0.66


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
importance_df = pd.DataFrame({
    'feature': features,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)
print(importance_df)

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


                  feature  importance_mean  importance_std
0         impressions_90d         0.046142        0.003481
4            avg_position         0.017048        0.003367
5                     ctr         0.013999        0.002510
2        content_age_days         0.012691        0.003653
1            sessions_90d         0.007688        0.002799
6              word_count        -0.005777        0.002266
3  days_since_last_update        -0.010204        0.002679


In [ ]:
test_df['model_score'] = model_scores
test_df['model_rank'] = test_df['model_score'].rank(ascending=False)
test_df['baseline_rank'] = test_df['baseline_score'].rank(ascending=False)
test_df['rank_gap'] = (test_df['baseline_rank'] - test_df['model_rank']).abs()

biggest_disagreements = test_df.sort_values('rank_gap', ascending=False)[
    ['content_id', 'model_score', 'baseline_score', 'model_rank', 'baseline_rank', 'trend_direction']
].head(10)
print(biggest_disagreements)

                 content_id  model_score  baseline_score  model_rank  \
2812   content_6b3520be1f95        0.025       14.190386      7077.5   
21468  content_96d8ccfc12c6        0.030       14.190386      7074.0   
13114  content_8f222654e93f        0.030       14.190386      7074.0   
24547  content_eed54a7e3211        0.030       14.190386      7074.0   
12941  content_de8e9909d824        0.040       14.190386      7068.5   
12982  content_fecb3b1efda1        0.045       14.190386      7064.0   
10449  content_e3a34c1706cf        0.045       14.190386      7064.0   
12679  content_54ddb72de5b2        0.050       14.190386      7061.5   
19995  content_93a0ca28cfe2        0.055       14.190386      7059.0   
22146  content_aa27d7969080        0.060       14.190386      7053.0   

       baseline_rank trend_direction  
2812           934.5            flat  
21468          934.5            flat  
13114          934.5            down  
24547          934.5             new  
12941       

Permutation importance: impressions_90d dominates (0.046), followed by
avg_position (0.017), ctr (0.014), and content_age_days (0.013). Notably,
days_since_last_update — the core signal behind my baseline's "stale" flag —
has a small NEGATIVE importance (-0.010), meaning shuffling it slightly
IMPROVED the model's score. The Random Forest essentially ignored staleness
and leaned almost entirely on impressions/position/CTR instead.

Biggest disagreements with the baseline: the 10 largest rank-gap cases all
show the model pushing pages the baseline liked (baseline_rank ~935, a
fairly high priority) down to near the bottom of its own ranking
(model_rank ~7070, out of ~7500 test rows). Looking at trend_direction for
these disagreement cases: only 2 of 10 are actually "down" — the rest are
flat, new, or stable. So on this particular sample, the model's
deprioritization was arguably more correct than the baseline's high ranking,
even though the model lost on the aggregate Precision@K metric.

Why the simple rule still won overall: my baseline directly encodes the
exact staleness-and-visibility and CTR-gap-by-tier interactions that the
label is correlated with, hand-tuned against real bucket tables from ML-07.
The Random Forest has to rediscover useful interactions from raw features
with much less direct signal, and with class_weight='balanced' on a
label that isn't a clean future outcome (is_declining_label is a same-window
proxy, not a true forward-looking label), it likely overfit to noise on
some features (word_count and days_since_last_update both show near-zero or
negative importance) rather than finding the baseline's sharper pattern.

Honest conclusion: on this metric, this split, and this proxy label, the
hand-written baseline beats Random Forest at Precision@20 and Precision@50.
This is a real, useful finding — not a failure to report quietly. It
suggests either (a) the starter dataset's proxy label is too weakly related
to the raw features for a generic model to beat a rule built with domain
knowledge of the exact same data, or (b) the model needs different/derived
features (e.g. an explicit stale_and_visible interaction term) rather than
raw columns, to have a fair shot at matching the baseline. Complexity alone
did not win here, and the baseline's simplicity remains the right choice to
ship as of this notebook.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- Compares against baseline on same data, same metric, same split ✓
- Valid split design: client-holdout, explained and verified (0 client
  overlap) ✓
- Method choice explained (Random Forest, why it fits the lane) ✓
- Useful metrics reported: Precision@20 and Precision@50 for both methods ✓
- Interpreted features (permutation importance) and errors (rank-gap
  disagreement analysis) ✓
- Does not reward complexity alone — baseline honestly reported as the
  stronger method on this metric ✓